# W13 · 선형 → 트리 · 깊이 선택 (노트북 02 · 약 2분)

**무엇을 하나** — 나무 깊이 1 ~ 12 의 학습 · 검증 RMSE(조건 그룹 5겹)와 학습 곡선을 그린다.

**왜 하나** — 학습 오차가 MD 반복 오차 아래로 내려가면 잡음을 외우는 중이다. 깊이는 검증 곡선과 반복 오차로 고른다 (덱 23 · 24 · 25장 · P13-3).

**실행** — 기본 = week13 폴더의 Claude Code 에 「`@notebooks/02_tree_depth.ipynb` 를 실행하고 결과를 요약해」 → week13 폴더에서 `python ../run_nb.py notebooks/02_tree_depth.ipynb` · 직접 보려면 `jupyter lab`.

In [ ]:
# 0) 작업 폴더 맞추기 — 로컬 · Claude Code 가 기본 · Colab 은 선택
#    run_nb.py 는 week13 에서 시작(그대로) · JupyterLab 은 notebooks/ 에서 시작 → week13 으로 이동 · Colab 이면 저장소를 받는다
#    scikit-learn 은 mdai 에 포함(../../environment.yml)
import os, pathlib
if not pathlib.Path('scripts/common.py').exists():
    if pathlib.Path('../scripts/common.py').exists():
        os.chdir('..')                       # 로컬: notebooks/ → week13
    else:                                     # Colab(선택): 저장소 받기
        !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo
        %cd repo/week13
print(os.getcwd())

In [ ]:
# 1) 깊이 곡선 — scripts/fit_linear_tree.py (약 30초) → out/tree_depth.json · out/tree2d.json
import sys, subprocess, json
import matplotlib.pyplot as plt
subprocess.run([sys.executable, 'scripts/fit_linear_tree.py'], check=True)
t = json.load(open('out/tree_depth.json'))
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (k, u) in zip(axs, [('density', 'g/cm3'), ('ced', 'MJ/m3'), ('bulk_modulus', 'GPa')]):
    g = t['targets'][k]['group']; dep = [r['depth'] for r in g]
    ax.plot(dep, [r['train_rmse'] for r in g], '--', label='train')
    ax.plot(dep, [r['val_rmse'] for r in g], '-o', ms=3, label='val (group)')
    ax.axhline(t['targets'][k]['md_seed_sd_median'], color='g', ls=':', label='MD seed SD')
    ax.set_title(f"{k} · best depth {t['targets'][k]['best']['depth']}"); ax.set_xlabel('depth'); ax.set_ylabel(f'RMSE ({u})')
axs[0].legend(fontsize=8); fig.tight_layout(); fig.savefig('out/tree_depth.png', dpi=120)
{k: v['best'] for k, v in t['targets'].items()}

In [ ]:
# 2) 학습 곡선 — scripts/learning_curve.py (약 30초) → out/learning_curve.json
subprocess.run([sys.executable, 'scripts/learning_curve.py'], check=True)
lc = json.load(open('out/learning_curve.json'))
print(json.dumps(lc, ensure_ascii=False)[:600])

## 확인
- 밀도 최소 깊이 8 · CED 6 · 체적탄성률 2 (검증 0.611 → 깊이 12 에서 0.625 GPa)
- 체적탄성률 학습 RMSE 가 0.483 까지 내려가 MD 반복 SD 0.436 에 다가간다 = 잡음을 외우는 중
- `out/tree_depth.png` 를 P13-3 결과에 첨부

못 돌아가면 `expected_output/tree_depth.json` 으로 P13-3 을 한다.